# VectorCam — Final metrics + Grad-CAM for BOTH models

Compares the two models:
- **baseline** = insectary only  (efficientvit_b0_baseline_insectary.pt)
- **augmented** = insectary + kenya_train  (efficientvit_b0_augmented_kenya.pt)

Does:
1. Exact metrics for each on the SAME held-out kenya_test (from kenya_test_split.csv).
2. Grad-CAM grids for EACH model separately, so you can show side-by-side that both
   attend to the mosquito (and how attention differs after seeing field data).

PREREQ: run the updated train_with_kenya.py first (it now SAVES both checkpoints +
the split CSVs). Put these in Drive:
  efficientvit_b0_baseline_insectary.pt, efficientvit_b0_augmented_kenya.pt,
  kenya_test_split.csv, master_seg.csv, data_seg.zip
GPU runtime.

## 0. Setup + data

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, zipfile, shutil, numpy as np, pandas as pd, torch
DRIVE='/content/drive/MyDrive/VectorCam'
for f in ['master_seg.csv','kenya_test_split.csv']:
    if not os.path.exists('/content/'+f): shutil.copy(os.path.join(DRIVE,f),'/content/'+f)
if not os.path.exists('/content/data_seg'):
    shutil.copy(os.path.join(DRIVE,'data_seg.zip'),'/content/data_seg.zip')
    with zipfile.ZipFile('/content/data_seg.zip') as z: z.extractall('/content/data_seg')
!pip -q install timm grad-cam scikit-learn
device='cuda' if torch.cuda.is_available() else 'cpu'; print(device)

## 1. Load master + the exact kenya_test held-out set

In [ ]:
m=pd.read_csv('/content/master_seg.csv')
def resolve(r):
    for c in ['seg_path','cropped_path']:
        if c in r and isinstance(r[c],str) and os.path.exists(r[c]): return r[c]
    return f"/content/data_seg/image_{int(r['ImageID'])}.jpg"
m['path']=m.apply(resolve,axis=1); m=m[m['path'].map(os.path.exists)].reset_index(drop=True)
CLASSES=["aedes_aegypti","anopheles_gambiae","anopheles_stephensi"]
c2i={c:i for i,c in enumerate(CLASSES)}; m['y']=m['species'].map(c2i)

ktest_ids=set(pd.read_csv('/content/kenya_test_split.csv')['ImageID'])
kenya_test=m[m['ImageID'].isin(ktest_ids)].reset_index(drop=True)
dev=m[m['drop']=='0623'].reset_index(drop=True)
print('kenya_test imgs:', len(kenya_test), '| device(0623) imgs:', len(dev))

## 2. Loader + metrics helpers

In [ ]:
import timm
from torch.utils.data import Dataset,DataLoader
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import classification_report, f1_score, accuracy_score
IMG=256; rs=int(IMG*1.14)
mean,std=[0.485,0.456,0.406],[0.229,0.224,0.225]
tf=T.Compose([T.Resize(rs),T.CenterCrop(IMG),T.ToTensor(),T.Normalize(mean,std)])
class DS(Dataset):
    def __init__(s,df): s.df=df.reset_index(drop=True)
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]; return tf(Image.open(r['path']).convert('RGB')), r['y'], i
def load_model(fname):
    ck=torch.load(os.path.join(DRIVE,fname),map_location=device)
    net=timm.create_model(ck['model'],pretrained=False,num_classes=3)
    net.load_state_dict(ck['state_dict']); net.to(device).eval()
    print('loaded',fname,'| trained_on:',ck.get('trained_on','?'))
    return net
def predict(net,df):
    dl=DataLoader(DS(df),batch_size=64,num_workers=2); P=np.zeros(len(df),dtype=int)
    with torch.no_grad():
        for x,y,idx in dl:
            with torch.amp.autocast('cuda'): o=net(x.to(device))
            P[idx.numpy()]=o.argmax(1).cpu().numpy()
    return P
def evalset(net,df,name):
    p=predict(net,df); y=df['y'].values
    print(f"\n=== {name} ===")
    print('macro-F1:',round(f1_score(y,p,average='macro'),4),'| acc:',round(accuracy_score(y,p),4))
    print(classification_report(y,p,target_names=CLASSES,digits=3,zero_division=0))

## 3. Metrics — baseline vs augmented on the SAME kenya_test

In [ ]:
base=load_model('efficientvit_b0_baseline_insectary.pt')
aug =load_model('efficientvit_b0_augmented_kenya.pt')
print('\n########## BASELINE (insectary only) ##########')
evalset(base, kenya_test, 'BASELINE -> kenya_test')
print('\n########## AUGMENTED (insectary + kenya_train) ##########')
evalset(aug, kenya_test, 'AUGMENTED -> kenya_test')

## 4. Grad-CAM grids — one per model
Run the same 12 kenya_test images through each model. Compare where each looks.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import matplotlib.pyplot as plt

samp=pd.concat([kenya_test[kenya_test.species==s].sample(min(4,len(kenya_test[kenya_test.species==s])),random_state=1)
                for s in CLASSES]).reset_index(drop=True)

def grid(net, title, fname):
    cam=GradCAM(model=net, target_layers=[net.stages[-1]])
    n=len(samp); cols=4; rows=(n+cols-1)//cols
    fig,axes=plt.subplots(rows,cols,figsize=(cols*3.2,rows*3.2)); axes=axes.flatten()
    for i,(_,r) in enumerate(samp.iterrows()):
        img=Image.open(r['path']).convert('RGB'); x=tf(img).unsqueeze(0).to(device)
        pred=net(x).argmax(1).item()
        g=cam(input_tensor=x,targets=[ClassifierOutputTarget(pred)])[0]
        disp=np.array(T.CenterCrop(IMG)(T.Resize(rs)(img))).astype(np.float32)/255
        ov=show_cam_on_image(disp,g,use_rgb=True)
        axes[i].imshow(ov); axes[i].axis('off')
        ok='OK' if pred==r['y'] else 'X'
        axes[i].set_title(f"T:{r['species'][:9]}\nP:{CLASSES[pred][:9]} {ok}",fontsize=7)
    for ax in axes[n:]: ax.axis('off')
    fig.suptitle(title,fontsize=11); plt.tight_layout()
    plt.savefig(os.path.join(DRIVE,fname),dpi=110); plt.show()
    print('saved',fname)

grid(base, 'BASELINE (insectary only) — Grad-CAM on kenya_test', 'gradcam_baseline.png')
grid(aug,  'AUGMENTED (insectary + kenya) — Grad-CAM on kenya_test', 'gradcam_augmented.png')

## Read
- Both grids: heat should sit on the mosquito (crop worked, no background shortcut).
- The augmented model should get MORE tiles correct (OK) on the same kenya_test images,
  especially stephensi — the visual proof of the +0.25 lift.
- Use both PNGs side by side in the submission.